# w0 — Statistical Bigram Detection

**Project**: Making Taste Legible: Symbolic Boundaries and Expert Valuation in Whisky Reviews  
**Purpose**: Detect statistically significant bigrams in the phrase-tokenized corpus (beyond curated phrases), present candidates for human review, then apply approved bigrams to the tokenized dataset.

## Workflow Overview

| Phase | Steps | What happens |
|-------|-------|--------------|
| **Phase 1** | 1–4 | Load data → tokenize → train gensim Phrases → filter and present candidates |
| **⏸ PAUSE** | — | Review candidate list; edit the `approved_bigrams` cell in Phase 2 |
| **Phase 2** | 5–6 | Apply approved bigrams → validate → save outputs |

## Prerequisites

These files must already exist (produced by `w0_preproc_phrases.ipynb`):
- `data/whiskyfun_tokenized.parquet`
- `data/whiskyfun_phrases.json`

---
# Phase 1 — Candidate Generation
---

## Step 1 — Setup and Data Loading

In [1]:
import re
import json
import shutil
from pathlib import Path
from collections import Counter

import pandas as pd
import nltk
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

DATA_DIR = Path('../data')
NOTEBOOK_DIR = Path('.')

TOKENIZED_PATH = DATA_DIR / 'whiskyfun_tokenized.parquet'
PHRASES_PATH   = DATA_DIR / 'whiskyfun_phrases.json'
BACKUP_PATH    = DATA_DIR / 'whiskyfun_tokenized_phrases_only.parquet'
BIGRAMS_PATH   = DATA_DIR / 'whiskyfun_bigrams.json'

TEXT_COLS = ['review_text', 'nose', 'mouth', 'finish', 'comments', 'nmf']

# --- Load data ---
assert TOKENIZED_PATH.exists(), f"Missing prerequisite: {TOKENIZED_PATH}"
assert PHRASES_PATH.exists(),   f"Missing prerequisite: {PHRASES_PATH}"

df = pd.read_parquet(TOKENIZED_PATH)
with open(PHRASES_PATH) as f:
    curated_phrases = json.load(f)

print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Curated phrases already applied: {len(curated_phrases)}")
print("\nSample tokenized review_text (first 3 rows):")
for i, text in enumerate(df['review_text'].dropna().head(3)):
    print(f"  [{i}] {text[:200]}")

Loaded dataset: 11,149 rows × 17 columns
Curated phrases already applied: 66

Sample tokenized review_text (first 3 rows):
  [0] it doesn't say it's a bruichladdich but it cannot be browmore, can it? colour: white wine. nose: the freshest freshness ever and a brininess that's even more obvious than usual. much more obvious… so 
  [1] from the very first distillation under the 'new regime'. colour: amber. nose: gunpowder all over the place at first nosing, then roasted chestnut and coffee. go on with corinthian raisin and coffee as
  [2] it's in 2009 that auction house bonham's sold eight unlabelled bottle of glenfiddich 1956, bottled by or for wine merchant thomas bucktrout & co. ltd. in guernsey. let's try one of those today, with t


## Step 2 — Tokenize into Word Lists

The tokenizer uses `re.findall(r'[a-z_]+', text.lower())` which preserves underscores so that already-joined curated phrases (e.g. `tropical_fruit`) are kept as single tokens during bigram detection.

In [2]:
def tokenize(text):
    """Lowercase and split on non-alphanumeric, preserving underscores."""
    if pd.isna(text):
        return []
    return re.findall(r'[a-z_]+', str(text).lower())

sentences = [tokenize(text) for text in df['review_text']]
# Drop empty sentences
sentences = [s for s in sentences if s]

all_tokens = [tok for sent in sentences for tok in sent]
vocab = set(all_tokens)

print(f"Sentences (reviews with text): {len(sentences):,}")
print(f"Total tokens:                  {len(all_tokens):,}")
print(f"Vocabulary size:               {len(vocab):,}")

# Confirm curated phrases appear as single tokens
sample_curated = curated_phrases[:5]
for phrase in sample_curated:
    count = sum(1 for sent in sentences if phrase in sent)
    print(f"  curated '{phrase}' present in {count:,} reviews")

Sentences (reviews with text): 11,149
Total tokens:                  1,935,546
Vocabulary size:               27,943
  curated 'old_bottle_effect' present in 31 reviews
  curated 'furniture_polish' present in 78 reviews
  curated 'pencil_shavings' present in 175 reviews
  curated 'tropical_fruit' present in 392 reviews
  curated 'milk_chocolate' present in 207 reviews


## Step 3 — Train gensim Phrases Model

In [3]:
bigram_model = Phrases(
    sentences,
    min_count=20,
    threshold=0.2,
    connector_words=ENGLISH_CONNECTOR_WORDS,
    scoring='npmi',
)

# Export all detected bigrams with NPMI scores
# gensim returns bytes keys; decode to str
raw_phrases = bigram_model.export_phrases()
bigram_scores = {}
for k, v in raw_phrases.items():
    key = k.decode() if isinstance(k, bytes) else k
    bigram_scores[key] = float(v)

candidates = sorted(bigram_scores.items(), key=lambda x: -x[1])
print(f"Total bigram candidates before filtering: {len(candidates):,}")

Total bigram candidates before filtering: 4,244


## Step 4 — Filter Candidates and Present for Review

Filters applied (in order):
1. Remove bigrams that overlap with existing curated phrases.
2. Remove collocations involving `with_water` (structural, not semantic).
3. Remove generic English collocations where both constituent words are stopwords.
4. Remove known generic filler patterns.
5. Tag remaining candidates by domain relevance.

In [4]:
# ---------------------------------------------------------------------------
# Filter 1: overlap with curated phrases
# ---------------------------------------------------------------------------
curated_set = set(curated_phrases)  # e.g. {'tropical_fruit', 'peat_smoke', ...}

def overlaps_curated(bigram, curated):
    """True if bigram is already in curated set or is a sub-phrase of one."""
    if bigram in curated:
        return True
    # Also check if constituent words form part of a longer curated phrase
    parts = set(bigram.split('_'))
    for phrase in curated:
        p_parts = set(phrase.split('_'))
        if parts == p_parts:
            return True
    return False

# ---------------------------------------------------------------------------
# Filter 2 & 3: with_water and stopword-only pairs
# ---------------------------------------------------------------------------
nltk_stopwords = set(stopwords.words('english'))
# ENGLISH_CONNECTOR_WORDS is a frozenset of connector words in gensim
all_stopwords = nltk_stopwords | set(ENGLISH_CONNECTOR_WORDS)

# Hard-coded generic filler patterns to always reject
FILLER_PATTERNS = {
    'a_lot', 'of_course', 'very_much', 'would_be', 'i_think',
    'as_well', 'so_far', 'at_least', 'a_bit', 'a_little',
    'quite_nice', 'very_good', 'very_nice', 'pretty_good',
    'not_quite', 'not_bad', 'not_sure', 'really_good',
    'make_sure', 'even_more', 'more_or', 'or_less',
    'in_the', 'on_the', 'of_the', 'at_the', 'to_the',
    'by_the', 'for_the', 'from_the', 'with_the',
    'and_the', 'or_the', 'but_the',
}

def is_generic(bigram):
    parts = bigram.split('_')
    if len(parts) != 2:
        return True
    w1, w2 = parts
    # both words are stopwords
    if w1 in all_stopwords and w2 in all_stopwords:
        return True
    # known filler
    if bigram in FILLER_PATTERNS:
        return True
    return False

def involves_with_water(bigram):
    return 'with_water' in bigram or bigram in {'with_water'}

# ---------------------------------------------------------------------------
# Domain vocabulary for tagging
# ---------------------------------------------------------------------------
# Words that signal a bigram is domain-relevant (sensory, structural, evaluative)
DOMAIN_WORDS = {
    # sensory / flavour
    'apple', 'apricot', 'banana', 'berry', 'blackberry', 'blackcurrant',
    'blueberry', 'cherry', 'citrus', 'coconut', 'coffee', 'cream', 'custard',
    'floral', 'fruity', 'ginger', 'grape', 'grapefruit', 'hazelnut', 'herb',
    'herbal', 'honey', 'jam', 'jasmine', 'lavender', 'leather', 'lemon',
    'lemongrass', 'lime', 'mango', 'maple', 'melon', 'menthol', 'mint',
    'nutmeg', 'oak', 'orange', 'peach', 'pear', 'pepper', 'peppercorn',
    'pine', 'plum', 'raspberry', 'raisin', 'rose', 'sandalwood', 'spice',
    'spicy', 'strawberry', 'tangerine', 'thyme', 'tobacco', 'tropical',
    'vanilla', 'walnut', 'wax', 'waxy', 'woody',
    # smoke / peat
    'ash', 'ashy', 'barbecue', 'bonfire', 'campfire', 'charcoal', 'coal',
    'earthen', 'earthy', 'iodine', 'medicinal', 'mossy', 'mud', 'muddy',
    'peat', 'peaty', 'phenol', 'phenolic', 'smoky', 'sooty', 'tar', 'tarry',
    # wood / cask
    'barrel', 'bourbon', 'cask', 'cedar', 'dunnage', 'hogshead', 'lumber',
    'mahogany', 'resinous', 'sawdust', 'sherry', 'stave', 'tannin', 'tannic',
    'woody',
    # bread / malt / grain
    'barley', 'biscuit', 'bread', 'brioche', 'cereal', 'cracker', 'dough',
    'grain', 'grainy', 'malt', 'malted', 'malty', 'oat', 'porridge', 'yeast',
    # confection / sweet
    'butterscotch', 'caramel', 'chocolate', 'confection', 'fudge', 'liquorice',
    'marmalade', 'nougat', 'praline', 'sugar', 'sugary', 'syrup', 'syrupy',
    'toffee', 'treacle',
    # floral / herbal
    'blossom', 'chamomile', 'floral', 'flower', 'grass', 'grassy', 'heather',
    'hay', 'mossy', 'rose', 'sappy', 'violet',
    # off-notes / flaws
    'acetic', 'acetone', 'damp', 'dusty', 'musty', 'rancid', 'rubbery',
    'solvent', 'sulphur', 'sulphurous', 'vegetal',
    # mouthfeel / texture
    'astringent', 'bitter', 'chalky', 'creamy', 'drying', 'fat', 'fatty',
    'grippy', 'oily', 'rich', 'silky', 'smooth', 'soft', 'velvety', 'viscous',
    # evaluative / complexity
    'balance', 'balanced', 'clean', 'complex', 'complexity', 'delicate',
    'depth', 'elegant', 'finesse', 'fresh', 'freshness', 'intense', 'layered',
    'lingering', 'mineral', 'minerality', 'refined', 'robust', 'round',
    'subtle', 'tight',
    # production / structural
    'aged', 'ancient', 'distillery', 'dunnage', 'fill', 'matured', 'old',
    'overaged', 'refill', 'underaged', 'vintage', 'year',
}

def domain_tag(bigram):
    parts = bigram.split('_')
    if any(p in DOMAIN_WORDS for p in parts):
        return 'keep'
    return 'uncertain'

# ---------------------------------------------------------------------------
# Compute corpus frequency (number of reviews containing the bigram)
# ---------------------------------------------------------------------------
def corpus_freq(bigram, sents):
    return sum(1 for sent in sents if bigram in sent)

# Pre-compute for efficiency: build per-sentence string sets
sent_sets = [set(s) for s in sentences]

def fast_freq(bigram):
    return sum(1 for s in sent_sets if bigram in s)

# ---------------------------------------------------------------------------
# Apply filters and build candidate table
# ---------------------------------------------------------------------------
filtered = []
for bigram, score in candidates:
    if overlaps_curated(bigram, curated_set):
        continue
    if involves_with_water(bigram):
        continue
    if is_generic(bigram):
        continue
    filtered.append((bigram, score))

print(f"After filtering: {len(filtered):,} candidates remain")

After filtering: 3,568 candidates remain


In [5]:
# Build display table for top 100 candidates
MAX_DISPLAY = 100
rows = []
for bigram, score in filtered[:MAX_DISPLAY]:
    freq = fast_freq(bigram)
    tag  = domain_tag(bigram)
    rows.append({'bigram': bigram, 'freq': freq, 'npmi': round(score, 4), 'tag': tag})

candidates_df = pd.DataFrame(rows).sort_values('freq', ascending=False).reset_index(drop=True)
print(f"Top {len(candidates_df)} candidates (sorted by corpus frequency):")
print(candidates_df.to_string(index=True))

Top 100 candidates (sorted by corpus frequency):
                    bigram  freq    npmi        tag
0                re_racked    41  0.8435  uncertain
1                beehive_y    27  0.8745  uncertain
2              extra_years    26  1.1279  uncertain
3              ultra_tight    21  0.9397       keep
4              ultra_zesty    21  0.8899  uncertain
5            ultra_precise    21  0.9412  uncertain
6                    ho_ho    16  1.0673  uncertain
7             aston_martin     8  1.0001  uncertain
8               west_coast     6  0.8830  uncertain
9                grape_pip     3  0.8651       keep
10               grand_cru     2  0.8700  uncertain
11              port_ellen     2  0.8751  uncertain
12             james_eadie     2  0.9209  uncertain
13               hong_kong     2  1.0074  uncertain
14           travel_retail     1  0.9484  uncertain
15              death_seat     1  0.8917  uncertain
16         sauvignon_blanc     1  0.9159  uncertain
17             

---
# ⏸ PAUSE — Human Review

**Review the candidate table above.** For each bigram decide: keep, reject, or uncertain.

Guidelines (from the project spec):
- **Keep** if the bigram is a genuine semantic unit in whisky tasting vocabulary (e.g. `apple_juice`, `wood_smoke`, `rubber_note`).
- **Reject** if it is generic English (`very_clean`, `quite_light`) or structural (`with_water`, `with_ice`).
- **Reject** production/metadata terms that are not sensory descriptors.
- **Trigrams are out of scope** — if you spot interesting trigrams, note them here but do not add to the approved list.

**After reviewing**, edit the `approved_bigrams` list in the first cell of Phase 2 below, then run Phase 2 from top to bottom.

---

---
# Phase 2 — Application

> **Edit `approved_bigrams` in the cell directly below, then run all cells in this section.**
---

## Step 5 — Approved Bigram List

Edit this cell with the bigrams you want to keep. Each entry is an underscore-joined string, e.g. `"apple_juice"`.

In [6]:
# ============================================================
#  EDIT THIS LIST — replace with your approved bigrams
# ============================================================
approved_bigrams = [
    # paste approved bigrams here, one per line, e.g.:
    # "apple_juice",
    # "wood_smoke",
    "cake_y",
    "beehive_y",
]
# ============================================================

print(f"Approved bigrams: {len(approved_bigrams)}")
for b in approved_bigrams:
    print(f"  {b}")

Approved bigrams: 2
  cake_y
  beehive_y


## Step 6 — Apply Approved Bigrams to All Text Columns

In [7]:
assert approved_bigrams, "approved_bigrams is empty — edit Step 5 before running Phase 2."

def phrase_to_pattern(phrase):
    """Case-insensitive word-boundary pattern; handles both singular and plural last word."""
    words = re.escape(phrase.replace('_', ' ')).replace(r'\ ', r'[\s_-]+')
    return re.compile(
        r'\b' + words + r'(?:e?s)?\b',
        re.IGNORECASE,
    )

# Apply longer bigrams first to avoid partial-match corruption
sorted_bigrams = sorted(approved_bigrams, key=lambda p: (len(p), p), reverse=True)
patterns = [(p, phrase_to_pattern(p)) for p in sorted_bigrams]

def apply_bigrams(text, compiled_patterns):
    if pd.isna(text):
        return text
    for phrase, pat in compiled_patterns:
        text = pat.sub(phrase, text)
    return text

# Reload the current (phrases-only) parquet so we apply on top of curated phrases
df_updated = pd.read_parquet(TOKENIZED_PATH)

for col in TEXT_COLS:
    if col in df_updated.columns:
        df_updated[col] = df_updated[col].apply(lambda t: apply_bigrams(t, patterns))

print("Bigrams applied to all text columns.")

Bigrams applied to all text columns.


In [8]:
# Frequency report for approved bigrams
print("Corpus frequency of approved bigrams (after application):")
print(f"{'bigram':<35} {'reviews':>8}")
print('-' * 45)

updated_sentences = [
    tokenize(t) for t in df_updated['review_text'] if pd.notna(t)
]
updated_sets = [set(s) for s in updated_sentences]

freq_report = []
for b in sorted_bigrams:
    n = sum(1 for s in updated_sets if b in s)
    freq_report.append((b, n))
    print(f"{b:<35} {n:>8,}")

Corpus frequency of approved bigrams (after application):
bigram                               reviews
---------------------------------------------
beehive_y                                 27
cake_y                                   108


## Step 7 — Validation

In [9]:
# Spot-check: show before/after for 5 reviews that contain at least one approved bigram
print("=== Spot-check: before/after bigram tokenization ===")
checked = 0
for idx, row in df.iterrows():
    original = str(row['review_text'])
    updated  = str(df_updated.at[idx, 'review_text'])
    if original != updated:
        print(f"\nRow {idx}:")
        print(f"  BEFORE: {original[:300]}")
        print(f"  AFTER:  {updated[:300]}")
        checked += 1
    if checked >= 5:
        break

if checked == 0:
    print("No differences found — check that approved_bigrams match text in the corpus.")

=== Spot-check: before/after bigram tokenization ===
No differences found — check that approved_bigrams match text in the corpus.


In [10]:
# Verify curated phrase tokens are not corrupted
print("=== Curated phrase integrity check ===")
sample_phrases = curated_phrases[:10]
for phrase in sample_phrases:
    space_form = phrase.replace('_', ' ')
    # Count occurrences of space-separated form (should be 0 — already joined)
    space_count = df_updated['review_text'].str.contains(
        r'\b' + re.escape(space_form) + r'\b', case=False, na=False, regex=True
    ).sum()
    token_count = df_updated['review_text'].str.contains(
        r'\b' + re.escape(phrase) + r'\b', case=False, na=False, regex=True
    ).sum()
    status = 'OK' if space_count == 0 else 'WARNING: space form still present'
    print(f"  {phrase:<35} token_count={token_count:,}  space_form_remaining={space_count}  [{status}]")

=== Curated phrase integrity check ===
  old_bottle_effect                   token_count=31  space_form_remaining=0  [OK]
  furniture_polish                    token_count=78  space_form_remaining=0  [OK]
  pencil_shavings                     token_count=175  space_form_remaining=0  [OK]
  tropical_fruit                      token_count=392  space_form_remaining=0  [OK]
  milk_chocolate                      token_count=207  space_form_remaining=0  [OK]
  dark_chocolate                      token_count=251  space_form_remaining=0  [OK]
  anti_maltoporn                      token_count=172  space_form_remaining=0  [OK]
  wet_cardboard                       token_count=15  space_form_remaining=0  [OK]
  ultra_classic                       token_count=61  space_form_remaining=0  [OK]
  toasted_bread                       token_count=90  space_form_remaining=0  [OK]


In [11]:
# Full frequency table: all bigrams (curated + approved) in final corpus
all_phrases = curated_phrases + approved_bigrams
print(f"{'phrase':<40} {'reviews':>8}")
print('-' * 50)
for phrase in sorted(all_phrases):
    n = df_updated['review_text'].str.contains(
        r'\b' + re.escape(phrase) + r'\b', case=False, na=False, regex=True
    ).sum()
    print(f"{phrase:<40} {n:>8,}")

phrase                                    reviews
--------------------------------------------------
anti_maltoporn                                172
barley_sugar                                  207
beehive_y                                      27
black_tea                                     332
blade_y                                        58
bourbon_cask                                   73
brown_sugar                                    76
burnt_rubber                                   14
cake_y                                        108
cane_sugar                                     26
chen_pi                                        21
citrus_fruit                                  229
cocoa_powder                                   97
dark_berry                                      1
dark_chocolate                                251
dark_fruit                                    199
direct_fired                                    9
dried_citrus                                   18

## Step 8 — Save Outputs

In [12]:
# 1. Back up pre-bigram tokenized file
shutil.copy2(TOKENIZED_PATH, BACKUP_PATH)
print(f"Backup saved: {BACKUP_PATH}")

# 2. Save bigram list
with open(BIGRAMS_PATH, 'w') as f:
    json.dump(sorted_bigrams, f, indent=2)
print(f"Bigrams saved: {BIGRAMS_PATH}  ({len(sorted_bigrams)} entries)")

# 3. Overwrite tokenized parquet with bigram-updated version
df_updated.to_parquet(TOKENIZED_PATH, index=False)
print(f"Updated tokenized dataset saved: {TOKENIZED_PATH}  ({len(df_updated):,} rows)")

print("\nDone. All acceptance criteria should now be met.")

Backup saved: ../data/whiskyfun_tokenized_phrases_only.parquet
Bigrams saved: ../data/whiskyfun_bigrams.json  (2 entries)
Updated tokenized dataset saved: ../data/whiskyfun_tokenized.parquet  (11,149 rows)

Done. All acceptance criteria should now be met.


---
## Acceptance Criteria Checklist

- [ ] `data/whiskyfun_tokenized_phrases_only.parquet` exists (backup)
- [ ] `data/whiskyfun_bigrams.json` exists with the approved bigram list
- [ ] Updated `data/whiskyfun_tokenized.parquet` includes both curated phrases and approved bigrams
- [ ] Notebook clearly separates candidate generation (Phase 1) from application (Phase 2)
- [ ] Frequency report for all approved bigrams is printed (Step 6)
- [ ] Spot-check confirms correct bigram tokenization without corrupting earlier phrase tokens (Step 7)